# 5. Action Clustering (Normalization)

This notebook normalizes the raw `Action (How was problem corrected?)` column into canonical word-level action patterns using HDBSCAN clustering + LLM labeling.

**Output:**
- `output/action_patterns.json` — list of canonical action patterns
- `output/action_mapping.csv` — raw action text → canonical pattern
- Updated `output/clustered_emr.csv` with `action_cluster_id` and `action_canonical` columns

In [1]:
import os
import sys
import json
import re
import numpy as np
import pandas as pd

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

# 2. Fallback to searching sys.path
if not project_root:
    for path in list(sys.path):
        if not path:
            continue
        candidate = os.path.abspath(path)
        for folder in [candidate, os.path.abspath(os.path.join(candidate, '..')), os.path.abspath(os.path.join(candidate, '..', '..'))]:
            if os.path.exists(os.path.join(folder, 'src', 'config.py')):
                project_root = folder
                break
        if project_root:
            break

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
    print(f"Current working directory: {cwd}")
    print(f"Relative path prefix: {path_prefix}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."

from src.config import settings
from src.utils import get_embeddings, get_llm

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("1. Loading Clustered Data...")
file_path = os.path.join(path_prefix, "output", "clustered_emr.csv")
if not os.path.exists(file_path):
    raise FileNotFoundError("clustered_emr.csv not found. Run 1_clustering.ipynb first.")

df = pd.read_csv(file_path)
print(f"Loaded {len(df)} records.")

# Extract and clean action column
action_col = None
for col in df.columns:
    if 'action' in col.lower() and 'corrected' in col.lower():
        action_col = col
        break
if action_col is None:
    # Try alternative column names
    for col in df.columns:
        if 'action' in col.lower():
            action_col = col
            break

if action_col is None:
    raise ValueError("Could not find Action column. Available columns: " + str(df.columns.tolist()))

print(f"Using action column: '{action_col}'")

# Clean actions
actions = df[action_col].fillna("").astype(str).str.strip()
# Filter out empty/very short actions
valid_mask = actions.str.len() > 2
print(f"Valid actions: {valid_mask.sum()} / {len(actions)}")

1. Loading Clustered Data...
Loaded 20630 records.
Using action column: 'Action (How was problem corrected?)'
Valid actions: 20413 / 20630


In [3]:
print("2. Embedding Action Texts...")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(settings.embedding_model)
action_texts = actions[valid_mask].tolist()

embeddings = model.encode(
    action_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {embeddings.shape}")

2. Embedding Action Texts...


Batches: 100%|██████████| 319/319 [01:07<00:00,  4.74it/s]

Embeddings shape: (20413, 384)


In [4]:
print("3. UMAP Dimensionality Reduction...")
import umap

reducer = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)
reduced = reducer.fit_transform(embeddings)
print(f"Reduced shape: {reduced.shape}")

3. UMAP Dimensionality Reduction...


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Reduced shape: (20413, 10)


In [5]:
print("4. HDBSCAN Clustering on Actions...")
import hdbscan

# Note: Increase min_cluster_size (e.g., 30 or 50) if you want fewer clusters and much faster LLM labeling
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=3,
    metric="euclidean",
    cluster_selection_method="eom",
)
action_labels = clusterer.fit_predict(reduced)

n_clusters = len(set(action_labels)) - (1 if -1 in action_labels else 0)
n_noise = int((action_labels == -1).sum())
print(f"Found {n_clusters} action clusters, {n_noise} noise ({n_noise/len(action_labels)*100:.1f}%)")

1. Loading Clustered Data...


NameError: name 'file10ath' is not defined

In [ ]:
print("5. LLM-Assisted Labeling of Action Clusters...")
from langchain_core.messages import HumanMessage

llm = get_llm(temperature=0.0)

action_cluster_labels = {}
unique_clusters = sorted(set(action_labels))
unique_clusters = [c for c in unique_clusters if c != -1]

for i, cid in enumerate(unique_clusters):
    mask = action_labels == cid
    cluster_texts = [action_texts[j] for j in range(len(action_texts)) if mask[j]]
    
    # Get representative samples (up to 5 closest to centroid)
    c_emb = embeddings[mask]
    centroid = c_emb.mean(axis=0)
    distances = np.linalg.norm(c_emb - centroid, axis=1)
    n_sel = min(5, len(cluster_texts))
    closest_idx = np.argsort(distances)[:n_sel]
    samples = [cluster_texts[idx] for idx in closest_idx]
    
    samples_text = "\n".join(f"  {j+1}. {t[:200]}" for j, t in enumerate(samples))
    
    prompt = (
        "Kamu adalah expert maintenance alat berat. "
        f"Berikut {len(samples)} contoh AKSI PERBAIKAN yang serupa "
        "(dikelompokkan oleh algoritma clustering):\n\n"
        f"{samples_text}\n\n"
        "Tugasmu:\n"
        "1. Analisis pola umum aksi perbaikan dari contoh-contoh di atas\n"
        "2. Berikan SATU label SINGKAT (2-4 kata, bahasa Inggris) yang "
        "menggambarkan aksi perbaikan ini di level kata (word-level), "
        "contoh: 'Replace Floating Seal', 'Tightening Bolt', 'Check & Inspection'\n\n"
        "PENTING: Jawab HANYA dalam format JSON:\n"
        '{"label": "Action Name", "confidence": 0.85}'
    )
    
    try:
        resp = llm.invoke([HumanMessage(content=prompt)])
        content = resp.content.strip()
        
        # Robust JSON extraction
        idx = content.find('{')
        if idx != -1:
            result, _ = json.JSONDecoder().raw_decode(content[idx:])
        else:
            result = json.loads(content)
            
        action_cluster_labels[cid] = result
        print(f"  [{i+1}/{len(unique_clusters)}] Cluster {cid} ({sum(mask)} items) -> {result.get('label')}")
    except Exception as e:
        action_cluster_labels[cid] = {"label": f"Action_Cluster_{cid}", "confidence": 0.0}
        print(f"  [{i+1}/{len(unique_clusters)}] Cluster {cid} -> FAILED: {e}")

print(f"\nLabeled {len(action_cluster_labels)} action clusters.")

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


In [ ]:
print("6. Building Outputs...")
output_dir = os.path.join(path_prefix, "output")
os.makedirs(output_dir, exist_ok=True)

# --- action_patterns.json ---
action_patterns = []
for cid in sorted(action_cluster_labels.keys()):
    mask = action_labels == cid
    info = action_cluster_labels[cid]
    action_patterns.append({
        "cluster_id": int(cid),
        "label": info["label"],
        "confidence": info.get("confidence", 0),
        "size": int(mask.sum()),
    })

action_patterns = sorted(action_patterns, key=lambda x: x["size"], reverse=True)
with open(os.path.join(output_dir, "action_patterns.json"), "w", encoding="utf-8") as f:
    json.dump(action_patterns, f, ensure_ascii=False, indent=2)
print(f"Saved {len(action_patterns)} action patterns to action_patterns.json")

# --- action_mapping.csv ---
mapping_data = []
for j, text in enumerate(action_texts):
    cid = action_labels[j]
    canonical = action_cluster_labels.get(cid, {}).get("label", "Uncategorized") if cid != -1 else "Uncategorized"
    mapping_data.append({"raw_action": text, "action_cluster_id": int(cid), "action_canonical": canonical})

mapping_df = pd.DataFrame(mapping_data)
mapping_df.to_csv(os.path.join(output_dir, "action_mapping.csv"), index=False, encoding="utf-8-sig")
print(f"Saved action_mapping.csv with {len(mapping_df)} rows")

# --- Update clustered_emr.csv ---
# Map back to original dataframe (including invalid/empty actions)
df["action_cluster_id"] = -1
df["action_canonical"] = "Uncategorized"

valid_indices = df.index[valid_mask]
for idx_pos, orig_idx in enumerate(valid_indices):
    cid = int(action_labels[idx_pos])
    df.at[orig_idx, "action_cluster_id"] = cid
    if cid != -1 and cid in action_cluster_labels:
        df.at[orig_idx, "action_canonical"] = action_cluster_labels[cid]["label"]

df.to_csv(file_path, index=False, encoding="utf-8-sig")
print(f"Updated clustered_emr.csv with action_cluster_id and action_canonical columns")

6. Building Outputs...
Saved 905 action patterns to action_patterns.json
Saved action_mapping.csv with 20413 rows
Updated clustered_emr.csv with action_cluster_id and action_canonical columns


In [ ]:
print("7. Summary...")
print(f"\nTop 10 Action Patterns by frequency:")
for p in action_patterns[:10]:
    print(f"  - {p['label']} ({p['size']} records, confidence: {p['confidence']})")

print(f"\nTotal action clusters: {len(action_patterns)}")
print(f"Uncategorized (noise): {n_noise} records")

7. Summary...

Top 10 Action Patterns by frequency:
  - Action_Cluster_460 (70 records, confidence: 0.0)
  - Action_Cluster_552 (64 records, confidence: 0.0)
  - Action_Cluster_3 (62 records, confidence: 0.0)
  - Action_Cluster_238 (59 records, confidence: 0.0)
  - Turbocharger Replacement (56 records, confidence: 1.0)
  - Action_Cluster_765 (54 records, confidence: 0.0)
  - Replace Parts (53 records, confidence: 1.0)
  - Injector Replacement (52 records, confidence: 1.0)
  - Process Claim Handling (51 records, confidence: 1.0)
  - Action_Cluster_50 (50 records, confidence: 0.0)

Total action clusters: 905
Uncategorized (noise): 3709 records
